In [ ]:
# Mount Google Drive (for Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except ImportError:
    IN_COLAB = False
    print("ℹ️  Not running in Google Colab - skipping drive mount")

# Set project root path
if IN_COLAB:
    PROJECT_ROOT = "/content/drive/MyDrive/face_based_attendance_system"
else:
    from pathlib import Path
    PROJECT_ROOT = str(Path("..").resolve())

print(f"📂 Project root: {PROJECT_ROOT}")

# 07 - Model Export

This notebook covers exporting the trained MobileFaceNet model for production deployment.

## Export Formats:
1. PyTorch (.pth) - for Python inference
2. TorchScript (.pt) - for C++ inference
3. ONNX (.onnx) - for cross-platform deployment
4. ONNX Runtime optimization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
import math
import time

print(f"PyTorch: {torch.__version__}")

# Check ONNX availability
try:
    import onnx
    import onnxruntime as ort
    ONNX_AVAILABLE = True
    print(f"ONNX: {onnx.__version__}")
    print(f"ONNX Runtime: {ort.__version__}")
except ImportError:
    ONNX_AVAILABLE = False
    print("ONNX not available. Install with: pip install onnx onnxruntime")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Model

In [ ]:
# MobileFaceNet (same as training)
class ECABlock(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        k = int(abs((math.log2(channels) + b) / gamma))
        k = k if k % 2 else k + 1
        k = max(3, k)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=k//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        y = self.avg_pool(x).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)


class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio, use_eca=True):
        super().__init__()
        hidden = in_ch * expand_ratio
        self.use_res = stride == 1 and in_ch == out_ch
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden)]
        layers += [nn.Conv2d(hidden, hidden, 3, stride, 1, groups=hidden, bias=False), nn.BatchNorm2d(hidden), nn.PReLU(hidden)]
        if use_eca:
            layers.append(ECABlock(hidden))
        layers += [nn.Conv2d(hidden, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.conv = nn.Sequential(*layers)
    
    def forward(self, x):
        return x + self.conv(x) if self.use_res else self.conv(x)


class MobileFaceNet(nn.Module):
    def __init__(self, embedding_dim=512, use_eca=True):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(3, 64, 3, 2, 1, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        self.conv2 = nn.Sequential(nn.Conv2d(64, 64, 3, 1, 1, groups=64, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        settings = [(2,64,5,2), (4,128,1,2), (2,128,6,1), (4,128,1,2), (2,128,2,1)]
        layers, in_ch = [], 64
        for exp, out, n, s in settings:
            for i in range(n):
                layers.append(InvertedResidual(in_ch, out, s if i==0 else 1, exp, use_eca))
                in_ch = out
        self.bottlenecks = nn.Sequential(*layers)
        self.conv3 = nn.Sequential(nn.Conv2d(128, 512, 1, bias=False), nn.BatchNorm2d(512), nn.PReLU(512))
        self.conv4 = nn.Sequential(nn.Conv2d(512, 512, 7, groups=512, bias=False), nn.BatchNorm2d(512))
        self.fc = nn.Linear(512, embedding_dim, bias=False)
        self.bn = nn.BatchNorm1d(embedding_dim)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bottlenecks(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.fc(x.flatten(1))
        x = self.bn(x)
        return F.normalize(x, p=2, dim=1)


# Load model
model = MobileFaceNet(embedding_dim=512, use_eca=True)

# Load weights if available
checkpoint_path = '../models/checkpoints/mobilefacenet_final.pth'
if Path(checkpoint_path).exists():
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    if 'model_state_dict' in state_dict:
        state_dict = state_dict['model_state_dict']
    model.load_state_dict(state_dict)
    print(f"Loaded weights from {checkpoint_path}")
else:
    print("Using random weights (no checkpoint found)")

model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Export to TorchScript

In [ ]:
def export_torchscript(model, output_path, example_input):
    """
    Export model to TorchScript format.
    
    TorchScript allows:
    - C++ inference without Python
    - Model optimization
    - Mobile deployment
    """
    model.eval()
    
    # Trace the model
    traced_model = torch.jit.trace(model, example_input)
    
    # Optimize for inference
    traced_model = torch.jit.optimize_for_inference(traced_model)
    
    # Save
    traced_model.save(output_path)
    print(f"TorchScript model saved to {output_path}")
    
    # Verify
    loaded_model = torch.jit.load(output_path)
    with torch.no_grad():
        original_output = model(example_input)
        loaded_output = loaded_model(example_input)
        diff = (original_output - loaded_output).abs().max().item()
        print(f"Max output difference: {diff:.8f}")
    
    return traced_model


# Export
output_dir = Path('../models/exported')
output_dir.mkdir(parents=True, exist_ok=True)

example_input = torch.randn(1, 3, 112, 112)
torchscript_model = export_torchscript(
    model,
    str(output_dir / 'mobilefacenet.pt'),
    example_input
)

# Check file size
ts_path = output_dir / 'mobilefacenet.pt'
print(f"TorchScript file size: {ts_path.stat().st_size / 1024 / 1024:.2f} MB")

## 3. Export to ONNX

In [ ]:
def export_onnx(model, output_path, example_input, opset_version=14):
    """
    Export model to ONNX format.
    
    ONNX enables:
    - Cross-platform deployment
    - Hardware acceleration (TensorRT, OpenVINO)
    - Web deployment (ONNX.js)
    """
    model.eval()
    
    # Export
    torch.onnx.export(
        model,
        example_input,
        output_path,
        export_params=True,
        opset_version=opset_version,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['embedding'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'embedding': {0: 'batch_size'}
        }
    )
    print(f"ONNX model saved to {output_path}")
    
    if ONNX_AVAILABLE:
        # Verify
        onnx_model = onnx.load(output_path)
        onnx.checker.check_model(onnx_model)
        print("ONNX model validation passed")
        
        # Test with ONNX Runtime
        session = ort.InferenceSession(output_path)
        ort_inputs = {'input': example_input.numpy()}
        ort_output = session.run(None, ort_inputs)[0]
        
        with torch.no_grad():
            torch_output = model(example_input).numpy()
        
        diff = np.abs(torch_output - ort_output).max()
        print(f"Max output difference (PyTorch vs ONNX): {diff:.8f}")


# Export
onnx_path = output_dir / 'mobilefacenet.onnx'
export_onnx(model, str(onnx_path), example_input)

# Check file size
print(f"ONNX file size: {onnx_path.stat().st_size / 1024 / 1024:.2f} MB")

## 4. Optimize ONNX Model

In [ ]:
def optimize_onnx(input_path, output_path):
    """
    Optimize ONNX model for inference.
    
    Optimizations:
    - Constant folding
    - Operator fusion
    - Quantization (optional)
    """
    if not ONNX_AVAILABLE:
        print("ONNX not available")
        return
    
    from onnxruntime.transformers import optimizer
    
    # Basic optimization
    optimized_model = optimizer.optimize_model(
        input_path,
        model_type='bert',  # General optimization
        num_heads=0,
        hidden_size=0
    )
    
    optimized_model.save_model_to_file(output_path)
    print(f"Optimized ONNX saved to {output_path}")
    
    # Compare sizes
    original_size = Path(input_path).stat().st_size
    optimized_size = Path(output_path).stat().st_size
    print(f"Size reduction: {(1 - optimized_size/original_size)*100:.1f}%")


# Try optimization (may require additional packages)
try:
    optimized_path = output_dir / 'mobilefacenet_optimized.onnx'
    optimize_onnx(str(onnx_path), str(optimized_path))
except Exception as e:
    print(f"Optimization skipped: {e}")

## 5. Benchmark Inference Speed

In [ ]:
def benchmark_pytorch(model, input_shape, num_runs=100, warmup=10):
    """Benchmark PyTorch inference."""
    model.eval()
    device = next(model.parameters()).device
    x = torch.randn(input_shape).to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)
    
    return np.mean(times), np.std(times)


def benchmark_onnx(onnx_path, input_shape, num_runs=100, warmup=10):
    """Benchmark ONNX Runtime inference."""
    if not ONNX_AVAILABLE:
        return 0, 0
    
    session = ort.InferenceSession(
        onnx_path,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )
    
    x = np.random.randn(*input_shape).astype(np.float32)
    
    # Warmup
    for _ in range(warmup):
        _ = session.run(None, {'input': x})
    
    # Benchmark
    times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        _ = session.run(None, {'input': x})
        times.append((time.perf_counter() - start) * 1000)
    
    return np.mean(times), np.std(times)


# Run benchmarks
print("\n=== Inference Speed Benchmark ===")
input_shape = (1, 3, 112, 112)

# PyTorch CPU
model_cpu = model.cpu()
mean_ms, std_ms = benchmark_pytorch(model_cpu, input_shape)
print(f"PyTorch CPU: {mean_ms:.2f} ± {std_ms:.2f} ms")

# PyTorch GPU
if torch.cuda.is_available():
    model_gpu = model.cuda()
    mean_ms, std_ms = benchmark_pytorch(model_gpu, input_shape)
    print(f"PyTorch GPU: {mean_ms:.2f} ± {std_ms:.2f} ms")

# ONNX Runtime
if ONNX_AVAILABLE:
    mean_ms, std_ms = benchmark_onnx(str(onnx_path), input_shape)
    print(f"ONNX Runtime: {mean_ms:.2f} ± {std_ms:.2f} ms")

## 6. Create Inference Wrapper

In [ ]:
class ONNXFaceRecognition:
    """
    Production-ready face recognition inference.
    
    Usage:
        recognizer = ONNXFaceRecognition('mobilefacenet.onnx')
        embedding = recognizer.get_embedding(image)
        similarity = recognizer.compare(emb1, emb2)
    """
    def __init__(self, onnx_path: str, use_gpu: bool = True):
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if use_gpu else ['CPUExecutionProvider']
        self.session = ort.InferenceSession(onnx_path, providers=providers)
        
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name
        
        print(f"Loaded ONNX model from {onnx_path}")
        print(f"Providers: {self.session.get_providers()}")
    
    def preprocess(self, image: np.ndarray) -> np.ndarray:
        """
        Preprocess image for inference.
        
        Args:
            image: RGB image (H, W, 3) or (112, 112, 3)
        
        Returns:
            Preprocessed tensor (1, 3, 112, 112)
        """
        import cv2
        
        # Resize if needed
        if image.shape[:2] != (112, 112):
            image = cv2.resize(image, (112, 112))
        
        # Normalize to [-1, 1]
        image = image.astype(np.float32) / 255.0
        image = (image - 0.5) / 0.5
        
        # HWC -> CHW
        image = image.transpose(2, 0, 1)
        
        # Add batch dimension
        image = np.expand_dims(image, 0)
        
        return image
    
    def get_embedding(self, image: np.ndarray) -> np.ndarray:
        """
        Get face embedding for an image.
        
        Args:
            image: RGB image (H, W, 3)
        
        Returns:
            512-dim L2-normalized embedding
        """
        preprocessed = self.preprocess(image)
        embedding = self.session.run(
            [self.output_name],
            {self.input_name: preprocessed}
        )[0]
        return embedding.flatten()
    
    def compare(self, emb1: np.ndarray, emb2: np.ndarray) -> float:
        """
        Compare two embeddings.
        
        Returns:
            Cosine similarity [-1, 1]
        """
        return float(np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2)))
    
    def verify(self, emb1: np.ndarray, emb2: np.ndarray, threshold: float = 0.4) -> bool:
        """
        Verify if two embeddings are from the same person.
        
        Args:
            threshold: Similarity threshold (default 0.4 for cosine)
        
        Returns:
            True if same person, False otherwise
        """
        return self.compare(emb1, emb2) >= threshold


# Test the wrapper
if ONNX_AVAILABLE:
    recognizer = ONNXFaceRecognition(str(onnx_path), use_gpu=False)
    
    # Test with random images
    img1 = np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
    img2 = np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
    
    emb1 = recognizer.get_embedding(img1)
    emb2 = recognizer.get_embedding(img2)
    
    print(f"Embedding shape: {emb1.shape}")
    print(f"Embedding norm: {np.linalg.norm(emb1):.4f} (should be ~1.0)")
    print(f"Similarity: {recognizer.compare(emb1, emb2):.4f}")

## 7. Export Summary

In [ ]:
print("\n=== Export Summary ===")
print(f"Output directory: {output_dir.absolute()}")
print("\nExported files:")

for f in output_dir.glob('*'):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.2f} MB")

print("\n=== Deployment Options ===")
print("1. TorchScript (.pt): Use with LibTorch (C++) or PyTorch")
print("2. ONNX (.onnx): Use with ONNX Runtime, TensorRT, OpenVINO")
print("\n=== Next Steps ===")
print("- Integrate with FastAPI backend")
print("- Create Docker container")
print("- Deploy to production")

## 8. Summary

This notebook exported the trained model to production-ready formats:

| Format | Size | Use Case |
|--------|------|----------|
| TorchScript | ~4MB | C++, Mobile |
| ONNX | ~4MB | Cross-platform, Web |

### Performance Targets:
- **CPU**: <20ms inference
- **GPU**: <5ms inference
- **Model size**: ~3.4MB (FP32)

### Integration:
The `ONNXFaceRecognition` wrapper provides a simple API:
```python
recognizer = ONNXFaceRecognition('mobilefacenet.onnx')
embedding = recognizer.get_embedding(aligned_face)
is_same = recognizer.verify(emb1, emb2, threshold=0.4)
```